# From machine identity to fault detection

Reproduce Tables I and III and the individual-model rows of Tables IV and V. The score archive contains the healthy training clips used for calibration and the held-out test clips. The small ID-count file records a separate healthy-identification evaluation. Run all cells on a CPU.

In [1]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
get_ipython().run_line_magic("run", '"notebooks/00_runtime.ipynb"')
import csv
import json
import numpy as np
from IPython.display import Markdown, display
from mimii.reproduce_results import fault_metrics

def table(columns, rows):
    lines = [columns, ["---"] * len(columns), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

with np.load("results/clip_scores.npz", allow_pickle=False) as archive:
    bundle = {k: archive[k] for k in archive.files if not k.endswith("_path")}
fits = json.loads(Path("results/fits.json").read_text())
with Path("results/healthy_id_counts.csv").open() as stream:
    id_counts = list(csv.DictReader(stream))
machines = ["fan", "pump", "slider", "valve"]

## Which recordings went into each split?

The source key keeps all three noise mixtures of an event together. Count training and test clips from the archive and validation clips from the audited ID denominators (Table I).

In [2]:
inventory = []
for machine in machines:
    tr = bundle["train_machine"] == machine
    te = bundle["test_machine"] == machine
    val = next(r for r in id_counts if r["model"] == "ResNet-18" and r["machine"] == machine)
    inventory.append([machine, tr.sum(), int(val["clips"]), (te & (bundle["test_label"] == 0)).sum(),
                      (te & (bundle["test_label"] == 1)).sum()])
table(["Machine", "Train (healthy)", "Validation (healthy)", "Test (healthy)", "Test (faulty)"],
      inventory + [["Total", *np.sum([r[1:] for r in inventory], axis=0)]])

| Machine | Train (healthy) | Validation (healthy) | Test (healthy) | Test (faulty) |
| --- | --- | --- | --- | --- |
| fan | 8565 | 1218 | 2442 | 4425 |
| pump | 7881 | 1119 | 2247 | 1368 |
| slider | 6744 | 954 | 1914 | 2670 |
| valve | 7758 | 1104 | 2211 | 1437 |
| Total | 30948 | 4395 | 8814 | 9900 |

## How well do the models identify a healthy machine?

Table III pools **correct IDs / clips**. Its CNN counts come from replaying the selected seed-42 checkpoints on validation clips. The two linear discriminant analysis (LDA) controls use healthy test clips. This cell recomputes the percentages from the audited counts.

In [3]:
id_rows = []
for model in dict.fromkeys(r["model"] for r in id_counts):
    counts = {r["machine"]: r for r in id_counts if r["model"] == model}
    correct = [int(counts[m]["correct"]) for m in machines]
    total = sum(int(r["clips"]) for r in counts.values())
    id_rows.append([model, counts["fan"]["split"], *correct, f"{sum(correct):,}/{total:,}",
                    f"{100 * sum(correct) / total:.4f}"])
table(["Model", "Split", *machines, "Pooled count", "ID accuracy (%)"], id_rows)

| Model | Split | fan | pump | slider | valve | Pooled count | ID accuracy (%) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| MobileNetV3-Small | validation | 1213 | 1103 | 937 | 1097 | 4,350/4,395 | 98.9761 |
| ResNet-18 | validation | 1218 | 1118 | 953 | 1104 | 4,393/4,395 | 99.9545 |
| EfficientNet-B0 | validation | 1218 | 1119 | 953 | 1104 | 4,394/4,395 | 99.9772 |
| Compact CNN | validation | 1218 | 1118 | 940 | 1104 | 4,380/4,395 | 99.6587 |
| Wide CNN | validation | 1218 | 1118 | 950 | 1104 | 4,390/4,395 | 99.8862 |
| LDA | healthy test | 2435 | 2229 | 1894 | 2105 | 8,663/8,814 | 98.2868 |
| Shuffled LDA | healthy test | 608 | 572 | 614 | 637 | 2,431/8,814 | 27.5811 |

## Can the scores detect a fault?

Use seed 42 for every main comparison. Fit the 95th-percentile cutoff on raw healthy training scores for each type and ID. AUC and standardized pAUC give equal weight to all 48 type/ID/SNR groups. Accuracy, recall and false-positive rate count individual test clips. The [ensemble notebook](04_ensemble_combinations.ipynb) computes the additional fusion rows.

In [4]:
metrics = {(fit["model"], fit["seed"]): fault_metrics(bundle, fit["key"]) for fit in fits}
rows = []
for (model, seed), r in metrics.items():
    if seed == 42:
        tp, fp, tn, fn = (r[k] for k in ("tp", "fp", "tn", "fn"))
        rates = [r["auc"], r["pauc"], (tp + tn) / (tp + fp + tn + fn),
                 tp / (tp + fn), fp / (fp + tn)]
        rows.append([model, tp, fp, *(f"{100 * rate:.2f}" for rate in rates)])
table(["Seed 42", "TP", "FP", "AUC (%)", "pAUC (%)", "Accuracy (%)", "Recall (%)", "FPR (%)"], rows)

| Seed 42 | TP | FP | AUC (%) | pAUC (%) | Accuracy (%) | Recall (%) | FPR (%) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| MobileNetV3-Small | 6232 | 1006 | 82.51 | 75.21 | 75.02 | 62.95 | 11.41 |
| ResNet-18 | 6928 | 553 | 90.47 | 84.21 | 81.16 | 69.98 | 6.27 |
| EfficientNet-B0 | 6869 | 584 | 89.68 | 84.33 | 80.68 | 69.38 | 6.63 |
| Compact CNN | 6582 | 614 | 89.29 | 82.14 | 78.99 | 66.48 | 6.97 |
| Compact mean-only | 6436 | 579 | 89.23 | 81.84 | 78.40 | 65.01 | 6.57 |
| Wide CNN | 6348 | 557 | 89.10 | 82.40 | 78.04 | 64.12 | 6.32 |
| CNN ensemble | 7592 | 720 | 93.62 | 88.79 | 83.82 | 76.69 | 8.17 |
| XGBoost | 7165 | 681 | 91.12 | 82.45 | 81.75 | 72.37 | 7.73 |

## How sensitive are the rankings to the training seed?

MobileNet and both custom CNNs also have seeds 43 and 44 on the identical recording split. Calculate the mean and population standard deviation across the three fits for the stability note under Table IV.

In [5]:
stability = []
for model in ("MobileNetV3-Small", "Compact CNN", "Wide CNN"):
    row = [model]
    for metric in ("auc", "pauc"):
        values = [100 * metrics[model, seed][metric] for seed in (42, 43, 44)]
        row.append(f"{np.mean(values):.2f} ± {np.std(values):.2f}")
    stability.append(row)
table(["Model", "AUC (%)", "pAUC (%)"], stability)

| Model | AUC (%) | pAUC (%) |
| --- | --- | --- |
| MobileNetV3-Small | 82.68 ± 0.15 | 74.89 ± 0.35 |
| Compact CNN | 88.40 ± 0.89 | 81.43 ± 0.58 |
| Wide CNN | 87.94 ± 1.36 | 82.06 ± 0.65 |